# GRPO Qwen3-0.6B calculator agent from the pinned SFT LoRA

Continue training the pinned SFT LoRA with GRPO on the reserved 600 prompts. The Qwen base remains frozen; the existing rank-16 SFT adapter is trainable. With nonzero KL, TRL creates a frozen copy of the initial SFT adapter as the reference policy.

The notebook is Colab-first and has two modes:

- `RUN_MODE=smoke` (default): 8 prompts, 4 generations, 10 optimizer steps, no Hub upload.
- `RUN_MODE=full`: all 600 reserved prompts for one epoch, then generate a separate upload helper.

Transformers generation is the verified T4 default. Colocated vLLM is available only as an explicit `USE_VLLM=1` experiment on L4/A100: vLLM 0.23's FlashInfer backend fails on the T4's SM75 kernels, so never enable it for an unmonitored full T4 run.

Upload the source notebook and immutable input files to `/content` before execution. Inject `.env` through the project Colab loader; never put tokens in the notebook. The eval and test splits are not loaded here.

In [ ]:
# @title Install the exact SFT-compatible stack plus colocated vLLM
import importlib.metadata
import os
import subprocess
import sys

# vLLM is opt-in because its FlashInfer backend fails on T4/SM75. On L4/A100,
# set USE_VLLM=1 before execution. TRL 1.8.0 supports vLLM <=0.23.0.
USE_VLLM_REQUESTED = os.environ.get("USE_VLLM", "0") == "1"
VLLM_VERSION = "0.23.0"
VLLM_VARIANT = "cu129"
VLLM_WHEEL_URL = (
    "https://github.com/vllm-project/vllm/releases/download/v0.23.0/"
    "vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
)
try:
    installed_vllm = importlib.metadata.version("vllm")
except importlib.metadata.PackageNotFoundError:
    installed_vllm = ""
installed_torch = importlib.metadata.version("torch")
if USE_VLLM_REQUESTED and not (
    installed_vllm.startswith(f"{VLLM_VERSION}+{VLLM_VARIANT}")
    and "+cu129" in installed_torch
):
    subprocess.run(
        [
            "uv", "pip", "install", "--system", "--reinstall", VLLM_WHEEL_URL,
            "--extra-index-url", "https://download.pytorch.org/whl/cu129",
            "--index-strategy", "unsafe-best-match",
        ],
        check=True,
    )
PINNED_PACKAGES = [
    "trl==1.8.0",
    "transformers==5.14.1",
    "peft==0.19.1",
    "datasets==5.0.0",
    "accelerate==1.14.0",
    "trackio==0.31.5",
    "torchao==0.17.0",
    # Required by TRL's experimental tool-response parser but not installed by TRL itself.
    "jmespath==1.0.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PINNED_PACKAGES],
    check=True,
)
PACKAGE_NAMES = [
    "trl", "transformers", "peft", "datasets", "accelerate", "trackio",
    "torchao", "jmespath",
]
if USE_VLLM_REQUESTED:
    PACKAGE_NAMES.append("vllm")
resolved = {name: importlib.metadata.version(name) for name in PACKAGE_NAMES}
expected = {
    item.split("==")[0]: item.split("==")[1]
    for item in PINNED_PACKAGES
}
assert {name: resolved[name] for name in expected} == expected, (resolved, expected)
if USE_VLLM_REQUESTED:
    assert resolved["vllm"].startswith(f"{VLLM_VERSION}+{VLLM_VARIANT}"), resolved["vllm"]
print("Resolved training stack:", resolved)
print("Restart the kernel if this cell changed already-imported packages.")

In [ ]:
# @title Authenticate, freeze identities, and validate immutable inputs
import hashlib
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import HfApi, create_bucket, login
from transformers import set_seed

# In an IPython/Colab kernel sys.stdout has no fileno(). vLLM's DEBUG path
# bypasses its fd-level stdout suppression and is therefore notebook-safe.
if os.environ.get("USE_VLLM", "0") == "1":
    os.environ.setdefault("VLLM_LOGGING_LEVEL", "DEBUG")

hf_token = os.environ.get("HF_TOKEN")
assert hf_token, "Inject the project .env with the Colab loader before execution."
login(token=hf_token, add_to_git_credential=False)
hf_api = HfApi(token=hf_token)
hf_user = hf_api.whoami()["name"]

SEED = 42
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
assert RUN_MODE in {"smoke", "full"}
USE_VLLM = os.environ.get("USE_VLLM", "0") == "1"
assert USE_VLLM == USE_VLLM_REQUESTED
ALLOW_MIXED_DATASET = os.environ.get("ALLOW_MIXED_DATASET", "0") == "1"
SMOKE_ROWS = int(os.environ.get("SMOKE_ROWS", "8"))
SMOKE_STEPS = int(os.environ.get("SMOKE_STEPS", "10"))
MAX_COMPLETION_LENGTH = int(os.environ.get("MAX_COMPLETION_LENGTH", "1024"))
MAX_TOOL_CALLING_ITERATIONS = int(os.environ.get("MAX_TOOL_CALLING_ITERATIONS", "6"))
assert MAX_COMPLETION_LENGTH >= 128
assert MAX_TOOL_CALLING_ITERATIONS >= 6
NUM_GENERATIONS = int(os.environ.get("NUM_GENERATIONS", "4"))
PER_DEVICE_TRAIN_BATCH_SIZE = int(os.environ.get("PER_DEVICE_TRAIN_BATCH_SIZE", "4"))
GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("GRADIENT_ACCUMULATION_STEPS", "2"))
STEPS_PER_GENERATION = int(os.environ.get("STEPS_PER_GENERATION", str(GRADIENT_ACCUMULATION_STEPS)))
VLLM_GPU_MEMORY_UTILIZATION = float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.30"))
LEGAL_PROGRESS_BONUS_MAX = float(os.environ.get("LEGAL_PROGRESS_BONUS_MAX", "0.0"))
LEGAL_PROGRESS_BONUS_PER_CALL = float(os.environ.get("LEGAL_PROGRESS_BONUS_PER_CALL", "0.0"))
STRICT_INVALID_REWARD = os.environ.get("STRICT_INVALID_REWARD", "0") == "1"
STRICT_VALID_CORRECT_REWARD = float(os.environ.get("STRICT_VALID_CORRECT_REWARD", "2.0"))
EXECUTION_BINARY_REWARD = os.environ.get("EXECUTION_BINARY_REWARD", "0") == "1"
assert PER_DEVICE_TRAIN_BATCH_SIZE % NUM_GENERATIONS == 0
assert GRADIENT_ACCUMULATION_STEPS % STEPS_PER_GENERATION == 0
assert 0 < VLLM_GPU_MEMORY_UTILIZATION < 1
assert 0 <= LEGAL_PROGRESS_BONUS_MAX <= 0.1
assert 0 <= LEGAL_PROGRESS_BONUS_PER_CALL <= 0.5
assert STRICT_VALID_CORRECT_REWARD >= 2.0
assert sum(bool(value) for value in (
    LEGAL_PROGRESS_BONUS_MAX,
    LEGAL_PROGRESS_BONUS_PER_CALL,
    STRICT_INVALID_REWARD,
    EXECUTION_BINARY_REWARD,
)) <= 1

BASE_ID = "Qwen/Qwen3-0.6B"
BASE_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
SFT_ADAPTER_ID = os.environ.get(
    "SFT_ADAPTER_ID", "tripathysagar/qwen3-0.6b-calc-sft200"
)
SFT_ADAPTER_REVISION = os.environ.get(
    "SFT_ADAPTER_REVISION", "86db06c14d91acc734e89a45bb5ca3ec4e1ee8f3"
)

DATA_DIR = Path("/content/data")
DATASET_PATH = DATA_DIR / "calculator_qwen3"
SFT_IDS_PATH = DATA_DIR / "calculator_qwen3_sft200_ids.txt"
GRPO_IDS_PATH = DATA_DIR / "calculator_qwen3_grpo600_ids.txt"
SOURCE_NOTEBOOK = Path("/content/grpo_qwen3_calculator.ipynb")
BACKEND_NAME = "vllm-colocate" if USE_VLLM else "transformers"
RUN_TAG = os.environ.get("RUN_TAG") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
assert RUN_TAG.replace("-", "").replace("_", "").isalnum(), RUN_TAG
RUN_ID = f"qwen3-calc-grpo600-{RUN_MODE}-{BACKEND_NAME}-seed{SEED}-{RUN_TAG}"
RUN_DIR = Path("/content/experiments") / RUN_ID
ADAPTER_DIR = RUN_DIR / "final_adapter"

HUB_REPO_ID = os.environ.get("GRPO_HUB_REPO_ID", f"{hf_user}/qwen3-0.6b-calc-grpo600")
TRACKIO_PROJECT = os.environ.get("GRPO_TRACKIO_PROJECT", "calc-rlvr-grpo")
TRACKIO_SPACE_ID = os.environ.get("GRPO_TRACKIO_SPACE_ID", f"{hf_user}/calc-rlvr-grpo")
TRACKIO_BUCKET_ID = os.environ.get("GRPO_TRACKIO_BUCKET_ID", f"{hf_user}/calc-rlvr-grpo-bucket")

EXPECTED_DATASET_FINGERPRINT = "6595b18449711494"
EXPECTED_SHA256 = {
    "sft200_ids": "acf5da4bf031cdc9e44b0aff3092e011710b959426c1925778bb70ec6ff125d8",
    "grpo600_ids": "33c87de11362e9b534f937e46e493a557f16b593219659f2792e81fccc59dbca",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for path in (SFT_IDS_PATH, GRPO_IDS_PATH, SOURCE_NOTEBOOK):
    assert path.is_file(), f"Upload required input to {path}"
assert DATASET_PATH.is_dir(), f"Upload the HF dataset directory to {DATASET_PATH}"
observed_hashes = {
    "sft200_ids": sha256_file(SFT_IDS_PATH),
    "grpo600_ids": sha256_file(GRPO_IDS_PATH),
}
assert observed_hashes == EXPECTED_SHA256, (observed_hashes, EXPECTED_SHA256)
assert hf_api.model_info(BASE_ID, revision=BASE_REVISION).sha == BASE_REVISION
assert hf_api.model_info(SFT_ADAPTER_ID, revision=SFT_ADAPTER_REVISION).sha == SFT_ADAPTER_REVISION

DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

bucket = create_bucket(TRACKIO_BUCKET_ID, private=True, exist_ok=True, token=hf_token)
assert bucket.bucket_id == TRACKIO_BUCKET_ID
print({
    "run_id": RUN_ID,
    "mode": RUN_MODE,
    "generation_backend": BACKEND_NAME,
    "output": str(RUN_DIR),
    "base_revision": BASE_REVISION,
    "sft_adapter_revision": SFT_ADAPTER_REVISION,
    "source_notebook_sha256": sha256_file(SOURCE_NOTEBOOK),
})

In [ ]:
# @title Build the reserved prompt-only GRPO dataset
from collections import Counter

from datasets import Dataset, load_from_disk

from calculator_semantics import remove_null_fields

hf_dataset = load_from_disk(str(DATASET_PATH))
observed_dataset_fingerprint = hf_dataset["train"]._fingerprint
if not ALLOW_MIXED_DATASET:
    assert observed_dataset_fingerprint == EXPECTED_DATASET_FINGERPRINT
train_rows = [remove_null_fields(dict(row)) for row in hf_dataset["train"]]
sft_ids = {line.strip() for line in SFT_IDS_PATH.read_text().splitlines() if line.strip()}
grpo_ids = {line.strip() for line in GRPO_IDS_PATH.read_text().splitlines() if line.strip()}

assert len(train_rows) == 800
assert len({row["id"] for row in train_rows}) == 800
assert len(sft_ids) == 200 and len(grpo_ids) == 600
assert not sft_ids & grpo_ids
assert sft_ids | grpo_ids == {row["id"] for row in train_rows}
assert all(row["metadata"]["split"] == "train" for row in train_rows)

grpo_rows = []
for row in train_rows:
    if row["id"] not in grpo_ids:
        continue
    grpo_rows.append({
        "prompt": [dict(message) for message in row["messages"][:2]],
        "id": row["id"],
        "expression": row["expression"],
        "final_answer": row["metadata"]["final_answer"],
        "expected_call_count": int(row["metadata"]["operation_count"]),
        "tier": row["metadata"]["tier"],
    })
grpo_rows.sort(key=lambda row: row["id"])
assert len(grpo_rows) == 600
assert all(len(row["prompt"]) == 2 for row in grpo_rows)

def smoke_slice(rows, total):
    quotas = {"easy": min(2, total), "medium": min(2, max(0, total - 2)), "hard": max(0, total - 4)}
    selected = []
    for tier, count in quotas.items():
        selected.extend([row for row in rows if row["tier"] == tier][:count])
    if len(selected) < total:
        used = {row["id"] for row in selected}
        remaining = [row for row in rows if row["id"] not in used]
        selected.extend(remaining[: total - len(selected)])
    return selected[:total]

selected_rows = smoke_slice(grpo_rows, SMOKE_ROWS) if RUN_MODE == "smoke" else grpo_rows
train_dataset = Dataset.from_list(selected_rows)
print({
    "rows": len(train_dataset),
    "tiers": dict(Counter(train_dataset["tier"])),
    "first_ids": train_dataset["id"][:5],
})

In [ ]:
# @title Define the stateful calculator environment and terminal reward
from statistics import mean
from typing import Any

import calculator_semantics
from calculator_semantics import CalculatorEnv, parse_final_answer, score_trajectory

def _content_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            part.get("text", "") for part in content
            if isinstance(part, dict) and part.get("type") == "text"
        )
    return "" if content is None else str(content)

def final_assistant_text(completion: Any) -> str:
    messages = completion if isinstance(completion, list) else [completion]
    for message in reversed(messages):
        if isinstance(message, dict) and message.get("role") == "assistant":
            return _content_text(message.get("content"))
    return ""

def apply_progress_bonus(
    score,
    expected_call_count,
    max_bonus=None,
    per_call_bonus=None,
    strict_invalid=None,
    execution_binary=None,
):
    if all(value is None for value in (
        max_bonus, per_call_bonus, strict_invalid, execution_binary
    )):
        max_bonus = LEGAL_PROGRESS_BONUS_MAX
        per_call_bonus = LEGAL_PROGRESS_BONUS_PER_CALL
        strict_invalid = STRICT_INVALID_REWARD
        execution_binary = EXECUTION_BINARY_REWARD
    else:
        max_bonus = 0.0 if max_bonus is None else max_bonus
        per_call_bonus = 0.0 if per_call_bonus is None else per_call_bonus
        strict_invalid = False if strict_invalid is None else strict_invalid
        execution_binary = False if execution_binary is None else execution_binary
    assert sum(bool(value) for value in (
        max_bonus, per_call_bonus, strict_invalid, execution_binary
    )) <= 1

    if execution_binary:
        binary_reward = 2.0 if (
            score["valid_trajectory"] and score["answer_correct"]
        ) else 0.0
        return binary_reward, binary_reward - score["reward"]

    if strict_invalid:
        if score["outcome"] == "no_answer":
            strict_reward = -1.2
        elif score["valid_trajectory"]:
            strict_reward = STRICT_VALID_CORRECT_REWARD if score["answer_correct"] else -0.5
        else:
            progress = min(score["legal_call_count"], expected_call_count) / expected_call_count
            strict_reward = -1.0 + 0.2 * progress
            reason = score["first_illegal_reason"]
            if reason == "post_completion_call":
                strict_reward -= 0.75
            elif reason is not None:
                strict_reward -= 0.5
            strict_reward -= 0.25 * score["overcall_count"]
            strict_reward = min(strict_reward, -0.5)
        return strict_reward, strict_reward - score["reward"]

    if score["outcome"] == "no_answer":
        return score["reward"], 0.0
    legal_call_count = min(score["legal_call_count"], expected_call_count)
    progress = legal_call_count / expected_call_count
    bonus = per_call_bonus * legal_call_count if per_call_bonus else max_bonus * progress
    return score["reward"] + bonus, bonus

def trajectory_reward(completions, environments, log_extra=None, log_metric=None, **kwargs):
    scores = []
    for completion, env in zip(completions, environments, strict=True):
        stated_answer = parse_final_answer(final_assistant_text(completion))
        scores.append(score_trajectory(
            expression=env.expression,
            calls=env.calls,
            stated_answer=stated_answer,
            expected_call_count=env.expected_call_count,
            correct_answer=env.correct_answer,
        ))

    actual_counts = [len(env.calls) for env in environments]
    expected_counts = [env.expected_call_count for env in environments]
    progress = [
        min(score["legal_call_count"], expected) / expected
        for score, expected in zip(scores, expected_counts, strict=True)
    ]
    reward_and_bonuses = [
        apply_progress_bonus(score, expected)
        for score, expected in zip(scores, expected_counts, strict=True)
    ]
    rewards = [value[0] for value in reward_and_bonuses]
    progress_bonuses = [value[1] for value in reward_and_bonuses]
    grouped_rollouts = {}
    for score, reward, env in zip(scores, rewards, environments, strict=True):
        group = grouped_rollouts.setdefault(env.id, {
            "tier": env.tier,
            "scores": [],
            "rewards": [],
        })
        group["scores"].append(score)
        group["rewards"].append(reward)
    if len(scores) > 1:
        assert all(
            len(group["scores"]) == NUM_GENERATIONS
            for group in grouped_rollouts.values()
        )
    rollout_groups = []
    for group in grouped_rollouts.values():
        successes = [
            score["outcome"] == "valid_correct" for score in group["scores"]
        ]
        rollout_groups.append({
            "tier": group["tier"],
            "all_success": all(successes),
            "all_failure": not any(successes),
            "mixed": any(successes) and not all(successes),
            "zero_reward_std": max(group["rewards"]) == min(group["rewards"]),
        })
    completion_ids = kwargs.get("completion_ids")
    if completion_ids is None:
        completion_ids = [[] for _ in scores]
    truncated = [len(ids) >= MAX_COMPLETION_LENGTH for ids in completion_ids]

    if log_extra is not None:
        log_extra("outcome", [score["outcome"] for score in scores])
        log_extra("tier", [env.tier for env in environments])
        log_extra("valid_trajectory", [score["valid_trajectory"] for score in scores])
        log_extra("answer_correct", [score["answer_correct"] for score in scores])
        log_extra("tool_call_count", actual_counts)
        log_extra("expected_tool_call_count", expected_counts)
        log_extra("legal_progress", progress)
        log_extra("reward_adjustment", progress_bonuses)
        log_extra("first_illegal_call_index", [
            score["first_illegal_call_index"] for score in scores
        ])
        log_extra("first_illegal_reason", [score["first_illegal_reason"] for score in scores])
        log_extra("truncated", truncated)

    if log_metric is not None and scores:
        for outcome in (
            "valid_correct", "valid_incorrect", "invalid_correct",
            "invalid_incorrect", "no_answer",
        ):
            log_metric(
                f"outcome/{outcome}_rate",
                mean(float(score["outcome"] == outcome) for score in scores),
            )
        log_metric("reward/task_success", mean(float(s["outcome"] == "valid_correct") for s in scores))
        log_metric("reward/no_answer_rate", mean(float(s["outcome"] == "no_answer") for s in scores))
        log_metric("reward/invalid_trace_rate", mean(float(not s["valid_trajectory"]) for s in scores))
        log_metric("reward/correct_but_invalid_rate", mean(
            float(s["answer_correct"] and not s["valid_trajectory"]) for s in scores
        ))
        log_metric("calls/undercall_rate", mean(
            float(actual < expected)
            for actual, expected in zip(actual_counts, expected_counts, strict=True)
        ))
        log_metric("calls/exact_rate", mean(
            float(actual == expected)
            for actual, expected in zip(actual_counts, expected_counts, strict=True)
        ))
        log_metric("calls/overcall_rate", mean(
            float(actual > expected)
            for actual, expected in zip(actual_counts, expected_counts, strict=True)
        ))
        log_metric("trace/legal_progress_mean", mean(progress))
        log_metric("reward/shaping_adjustment_mean", mean(progress_bonuses))
        log_metric("reward/nonnegative_invalid_rate", mean(
            float((not score["valid_trajectory"]) and reward >= 0)
            for score, reward in zip(scores, rewards, strict=True)
        ))
        for label in ("all_success", "all_failure", "mixed", "zero_reward_std"):
            log_metric(
                f"groups/{label}_rate",
                mean(float(group[label]) for group in rollout_groups),
            )
        for tier in ("easy", "medium", "hard"):
            tier_groups = [group for group in rollout_groups if group["tier"] == tier]
            if tier_groups:
                log_metric(
                    f"groups/{tier}_all_failure_rate",
                    mean(float(group["all_failure"]) for group in tier_groups),
                )
                log_metric(
                    f"groups/{tier}_zero_reward_std_rate",
                    mean(float(group["zero_reward_std"]) for group in tier_groups),
                )
        log_metric("completion/truncated_rate", mean(float(value) for value in truncated))
        for tier in ("easy", "medium", "hard"):
            tier_scores = [
                score for score, env in zip(scores, environments, strict=True)
                if env.tier == tier
            ]
            if tier_scores:
                log_metric(
                    f"tier/{tier}_task_success",
                    mean(float(score["outcome"] == "valid_correct") for score in tier_scores),
                )
    return rewards

In [ ]:
# @title Unit-test every reward outcome and semantic failure class
def check(expression, calls, answer, expected, correct, reward, outcome):
    result = score_trajectory(expression, calls, answer, expected, correct)
    assert result["reward"] == reward, result
    assert result["outcome"] == outcome, result
    return result

valid_calls = [
    {"op": "*", "a": 3, "b": 4},
    {"op": "+", "a": 2, "b": 12},
]
invalid_calls = [
    {"op": "+", "a": 2, "b": 3},  # precedence violation
    {"op": "*", "a": 5, "b": 4},
]
check("2+3*4", valid_calls, 14, 2, 14, 2.0, "valid_correct")
check("2+3*4", valid_calls, 13, 2, 14, 0.0, "valid_incorrect")
check("2+3*4", invalid_calls, 14, 2, 14, 0.0, "invalid_correct")
check("2+3*4", invalid_calls, 20, 2, 14, -1.0, "invalid_incorrect")
check("2+3*4", valid_calls, None, 2, 14, -0.1, "no_answer")

full_score = score_trajectory("2+3*4", valid_calls, 14, 2, 14)
partial_score = score_trajectory("2+3*4", valid_calls[:1], 14, 2, 14)
invalid_score = score_trajectory("2+3*4", [valid_calls[0], valid_calls[0]], 20, 2, 14)
no_answer_score = score_trajectory("2+3*4", valid_calls, None, 2, 14)
assert apply_progress_bonus(full_score, 2, 0.1) == (2.1, 0.1)
assert apply_progress_bonus(partial_score, 2, 0.1) == (0.05, 0.05)
assert apply_progress_bonus(invalid_score, 2, 0.1) == (-0.95, 0.05)
assert apply_progress_bonus(no_answer_score, 2, 0.1) == (-0.1, 0.0)
for expected in (1, 3, 5):
    normalized_score = {"reward": 0.0, "outcome": "valid_incorrect", "legal_call_count": expected}
    shaped_reward, bonus = apply_progress_bonus(normalized_score, expected, 0.1)
    assert abs(shaped_reward - 0.1) < 1e-12 and abs(bonus - 0.1) < 1e-12
prefix_score = {"reward": 0.0, "outcome": "invalid_correct", "legal_call_count": 1}
shaped_reward, bonus = apply_progress_bonus(prefix_score, 5, 0.1)
assert abs(shaped_reward - 0.02) < 1e-12 and abs(bonus - 0.02) < 1e-12
assert apply_progress_bonus(full_score, 2, per_call_bonus=0.5) == (3.0, 1.0)
assert apply_progress_bonus(partial_score, 2, per_call_bonus=0.5) == (0.5, 0.5)
assert apply_progress_bonus(invalid_score, 2, per_call_bonus=0.5) == (-0.5, 0.5)
assert apply_progress_bonus(no_answer_score, 2, per_call_bonus=0.5) == (-0.1, 0.0)
hard_score = {"reward": 0.0, "outcome": "valid_incorrect", "legal_call_count": 5}
assert apply_progress_bonus(hard_score, 5, per_call_bonus=0.5) == (2.5, 2.5)

valid_wrong_score = score_trajectory("2+3*4", valid_calls, 13, 2, 14)
precedence_score = score_trajectory("2+3*4", invalid_calls, 20, 2, 14)
post_completion_score = score_trajectory(
    "2+3", [{"op": "+", "a": 3, "b": 2}, {"op": "+", "a": 5, "b": 2}], 7, 1, 5
)
assert apply_progress_bonus(full_score, 2, strict_invalid=True)[0] == STRICT_VALID_CORRECT_REWARD
assert apply_progress_bonus(valid_wrong_score, 2, strict_invalid=True)[0] == -0.5
assert apply_progress_bonus(no_answer_score, 2, strict_invalid=True)[0] == -1.2
assert abs(apply_progress_bonus(partial_score, 2, strict_invalid=True)[0] - (-0.9)) < 1e-12
assert apply_progress_bonus(precedence_score, 2, strict_invalid=True)[0] == -1.5
assert apply_progress_bonus(post_completion_score, 1, strict_invalid=True)[0] == -1.8
assert apply_progress_bonus(full_score, 2, execution_binary=True)[0] == 2.0
assert apply_progress_bonus(valid_wrong_score, 2, execution_binary=True)[0] == 0.0
assert apply_progress_bonus(precedence_score, 2, execution_binary=True)[0] == 0.0
assert apply_progress_bonus(no_answer_score, 2, execution_binary=True)[0] == 0.0
assert check(
    "2+3", [{"op": "+", "a": 3, "b": 2}], 5, 1, 5, 2.0, "valid_correct"
)["valid_trajectory"]
repeated_literal_score = score_trajectory(
    "6*5+6",
    [{"op": "*", "a": 6, "b": 5}, {"op": "+", "a": 30, "b": 6}],
    36,
    2,
    36,
)
assert repeated_literal_score["valid_trajectory"]
shortcut_score = score_trajectory(
    "2+3", [{"op": "+", "a": 2, "b": 3}, {"op": "+", "a": 5, "b": 0}], 5, 1, 5
)
assert shortcut_score["first_illegal_reason"] == "post_completion_call"

alternative_order = [
    {"op": "*", "a": 4, "b": 5},
    {"op": "*", "a": 2, "b": 3},
    {"op": "+", "a": 6, "b": 20},
]
assert check("2*3+4*5", alternative_order, 26, 3, 26, 2.0, "valid_correct")["valid_trajectory"]

failure_cases = {
    "omitted": valid_calls[:1],
    "duplicate": [valid_calls[0], valid_calls[0], valid_calls[1]],
    "wrong_operand": [{"op": "*", "a": 3, "b": 5}, valid_calls[1]],
    "wrong_operator": [{"op": "+", "a": 3, "b": 4}, valid_calls[1]],
}
for name, calls in failure_cases.items():
    result = score_trajectory("2+3*4", calls, 99, 2, 14)
    assert not result["valid_trajectory"], (name, result)

probe = CalculatorEnv()
probe.reset(id="probe", expression="2+3", final_answer=5, expected_call_count=1, tier="easy")
assert probe.calculator("+", 2, 3) == '{"result": 5}'
assert probe.calls == [{"op": "+", "a": 2, "b": 3}]

logged_metrics, logged_columns = {}, {}
def capture_metric(name, value):
    logged_metrics[name] = value
def capture_column(name, values):
    logged_columns[name] = values
probe_reward = trajectory_reward(
    completions=[[{"role": "assistant", "content": "The answer is 5."}]],
    environments=[probe],
    completion_ids=[[1, 2, 3]],
    log_metric=capture_metric,
    log_extra=capture_column,
)
configured_probe_adjustment = (
    STRICT_VALID_CORRECT_REWARD - 2.0
    if STRICT_INVALID_REWARD
    else LEGAL_PROGRESS_BONUS_MAX + LEGAL_PROGRESS_BONUS_PER_CALL
)
assert probe_reward == [2.0 + configured_probe_adjustment]
assert logged_metrics["tier/easy_task_success"] == 1.0
assert logged_metrics["calls/exact_rate"] == 1.0
assert logged_metrics["trace/legal_progress_mean"] == 1.0
assert logged_metrics["reward/shaping_adjustment_mean"] == configured_probe_adjustment
assert logged_metrics["reward/nonnegative_invalid_rate"] == 0.0
assert logged_metrics["completion/truncated_rate"] == 0.0
assert logged_columns["reward_adjustment"] == [configured_probe_adjustment]
assert logged_columns["outcome"] == ["valid_correct"]
print("Reward, metric, and environment unit tests passed.")

In [ ]:
# @title Load the pinned base and attach the SFT LoRA as trainable
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA runtime is required. Select a T4 or L4 in Colab.")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
if USE_VLLM and capability[0] < 8:
    raise RuntimeError(
        "vLLM is disabled on T4/SM75 because FlashInfer fails at runtime; "
        "use Transformers generation or select an L4/A100."
    )
use_bf16 = capability[0] >= 8 and torch.cuda.is_bf16_supported()
torch_dtype = torch.bfloat16 if use_bf16 else torch.float16
dtype_name = "bfloat16" if use_bf16 else "float16"

tokenizer = AutoTokenizer.from_pretrained(
    SFT_ADAPTER_ID,
    revision=SFT_ADAPTER_REVISION,
    token=hf_token,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    revision=BASE_REVISION,
    token=hf_token,
    dtype=torch_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
policy = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER_ID,
    revision=SFT_ADAPTER_REVISION,
    token=hf_token,
    is_trainable=True,
)
policy.config.use_cache = False

sft_lora = policy.peft_config["default"]
assert sft_lora.r == 16 and sft_lora.lora_alpha == 32
assert set(sft_lora.target_modules) == {
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
}
assert not getattr(policy, "is_loaded_in_4bit", False)
policy.print_trainable_parameters()
print({"gpu": gpu_name, "capability": capability, "dtype": dtype_name, "merged": False})

In [ ]:
# @title Build GRPOTrainer and verify the frozen SFT reference adapter
from trl import GRPOConfig, GRPOTrainer

GRPO_KWARGS = dict(
    output_dir=str(RUN_DIR),
    seed=SEED,
    data_seed=SEED,
    shuffle_dataset=True,
    remove_unused_columns=False,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    steps_per_generation=STEPS_PER_GENERATION,
    learning_rate=1e-5,
    num_train_epochs=1,
    max_steps=SMOKE_STEPS if RUN_MODE == "smoke" else -1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
    max_grad_norm=1.0,
    beta=0.02,
    loss_type="dapo",
    scale_rewards="batch",
    temperature=0.8,
    top_p=0.95,
    top_k=20,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_tool_calling_iterations=MAX_TOOL_CALLING_ITERATIONS,
    mask_truncated_completions=False,
    chat_template_kwargs={"enable_thinking": True},
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    do_eval=False,
    eval_strategy="no",
    logging_steps=1,
    save_strategy="steps",
    save_steps=5 if RUN_MODE == "smoke" else 25,
    save_total_limit=2,
    report_to="trackio",
    project=TRACKIO_PROJECT,
    run_name=RUN_ID,
    trackio_space_id=TRACKIO_SPACE_ID,
    trackio_bucket_id=TRACKIO_BUCKET_ID,
    trackio_static_space_id=False,
    hub_private_repo=True,
    # A Colab session exposes one GPU, so use colocate rather than server mode.
    use_vllm=USE_VLLM,
    vllm_mode="colocate",
    vllm_model_impl="vllm",
    vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=2048,
    vllm_tensor_parallel_size=1,
    vllm_enable_sleep_mode=False,
    vllm_importance_sampling_correction=True,
)
grpo_config = GRPOConfig(**GRPO_KWARGS)
trainer = GRPOTrainer(
    model=policy,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=trajectory_reward,
    environment_factory=CalculatorEnv,
)

def adapter_state_hash(model, adapter_name: str) -> str:
    digest = hashlib.sha256()
    marker = f".{adapter_name}."
    matched = 0
    for name, parameter in sorted(model.named_parameters()):
        if marker not in name:
            continue
        digest.update(name.replace(marker, ".ADAPTER.").encode())
        raw = parameter.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes()
        digest.update(raw)
        matched += 1
    assert matched > 0, f"No parameters found for adapter {adapter_name}"
    return digest.hexdigest()

assert "default" in trainer.model.peft_config
assert "ref" in trainer.model.peft_config, "beta>0 must preserve the initial SFT adapter as ref"
trainable_names = [name for name, parameter in trainer.model.named_parameters() if parameter.requires_grad]
assert trainable_names and all(".default." in name and "lora_" in name for name in trainable_names)
ref_names = [name for name, _ in trainer.model.named_parameters() if ".ref." in name]
assert ref_names and all(not dict(trainer.model.named_parameters())[name].requires_grad for name in ref_names)
initial_default_hash = adapter_state_hash(trainer.model, "default")
initial_ref_hash = adapter_state_hash(trainer.model, "ref")
assert initial_default_hash == initial_ref_hash, "Reference must begin as an exact SFT adapter copy"
print({
    "trainable_parameter_tensors": len(trainable_names),
    "reference_parameter_tensors": len(ref_names),
    "initial_sft_adapter_sha256": initial_default_hash,
    "rows": len(train_dataset),
    "max_steps": grpo_config.max_steps,
    "generation_backend": BACKEND_NAME,
    "num_generations": NUM_GENERATIONS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "steps_per_generation": STEPS_PER_GENERATION,
    "generation_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * STEPS_PER_GENERATION,
    "prompts_per_microbatch": PER_DEVICE_TRAIN_BATCH_SIZE // NUM_GENERATIONS,
    "prompts_per_optimizer_step": (
        PER_DEVICE_TRAIN_BATCH_SIZE // NUM_GENERATIONS
    ) * GRADIENT_ACCUMULATION_STEPS,
    "completions_per_optimizer_step": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "legal_progress_bonus_max": LEGAL_PROGRESS_BONUS_MAX,
    "legal_progress_bonus_per_call": LEGAL_PROGRESS_BONUS_PER_CALL,
    "strict_invalid_reward": STRICT_INVALID_REWARD,
    "strict_valid_correct_reward": STRICT_VALID_CORRECT_REWARD,
    "execution_binary_reward": EXECUTION_BINARY_REWARD,
    "vllm_gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION if USE_VLLM else None,
})

In [ ]:
# @title Verify the trainer-generated calculator schema and Qwen tool template
assert len(trainer.tools) == 1
assert trainer.tools[0].__name__ == "calculator"
rendered_prompt = tokenizer.apply_chat_template(
    train_dataset[0]["prompt"],
    tools=trainer.tools,
    chat_template=trainer.chat_template,
    add_generation_prompt=True,
    tokenize=False,
    **grpo_config.chat_template_kwargs,
)
assert "calculator" in rendered_prompt
assert all(symbol in rendered_prompt for symbol in ('"+"', '"-"', '"*"'))
assert "<think>" in rendered_prompt or "<|im_start|>assistant" in rendered_prompt
print(rendered_prompt[:2000])

In [ ]:
# @title Train, enforce smoke gates, and save reproducibility artifacts
import math
import platform
import shutil
import subprocess
import threading
import time


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, set):
        return sorted(value)
    if hasattr(value, "value"):
        return value.value
    return str(value)


def write_json_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=json_default) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def gpu_snapshot() -> dict[str, float | str]:
    output = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=memory.used,memory.total,utilization.gpu,power.draw",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip().splitlines()[0]
    used, total, utilization, power = (float(value.strip()) for value in output.split(","))
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "memory_used_mib": used,
        "memory_total_mib": total,
        "memory_free_mib": total - used,
        "gpu_utilization_percent": utilization,
        "power_draw_watts": power,
    }


gpu_samples = [gpu_snapshot()]
gpu_monitor_stop = threading.Event()

def monitor_gpu() -> None:
    while not gpu_monitor_stop.wait(0.5):
        try:
            gpu_samples.append(gpu_snapshot())
        except (OSError, subprocess.SubprocessError, ValueError):
            pass

gpu_monitor = threading.Thread(target=monitor_gpu, name="gpu-telemetry", daemon=True)
gpu_monitor.start()
started_at = datetime.now(timezone.utc).isoformat()
try:
    train_result = trainer.train()
finally:
    gpu_monitor_stop.set()
    gpu_monitor.join(timeout=5)
    gpu_samples.append(gpu_snapshot())
ended_at = datetime.now(timezone.utc).isoformat()
trainer.save_state()

gpu_utilizations = [sample["gpu_utilization_percent"] for sample in gpu_samples]
gpu_telemetry_summary = {
    "sample_interval_seconds": 0.5,
    "num_samples": len(gpu_samples),
    "memory_total_mib": max(sample["memory_total_mib"] for sample in gpu_samples),
    "peak_memory_used_mib": max(sample["memory_used_mib"] for sample in gpu_samples),
    "minimum_memory_free_mib": min(sample["memory_free_mib"] for sample in gpu_samples),
    "mean_gpu_utilization_percent": float(np.mean(gpu_utilizations)),
    "p95_gpu_utilization_percent": float(np.percentile(gpu_utilizations, 95)),
    "maximum_gpu_utilization_percent": max(gpu_utilizations),
    "mean_power_draw_watts": float(np.mean([sample["power_draw_watts"] for sample in gpu_samples])),
    "maximum_power_draw_watts": max(sample["power_draw_watts"] for sample in gpu_samples),
}
write_json_atomic(RUN_DIR / "gpu_telemetry.json", {
    "run_id": RUN_ID,
    "summary": gpu_telemetry_summary,
    "samples": gpu_samples,
})

for entry in trainer.state.log_history:
    for key, value in entry.items():
        if isinstance(value, (int, float)) and not math.isfinite(value):
            raise RuntimeError(f"Non-finite metric: {key}={value}")

final_default_hash = adapter_state_hash(trainer.model, "default")
final_ref_hash = adapter_state_hash(trainer.model, "ref")
assert final_default_hash != initial_default_hash, "GRPO did not update the trainable LoRA"
assert final_ref_hash == initial_ref_hash, "Frozen SFT reference adapter changed"
reward_stds = [
    entry["reward_std"] for entry in trainer.state.log_history
    if "reward_std" in entry
]
assert reward_stds and any(value > 0 for value in reward_stds), (
    "No within-generation reward contrast; GRPO has no learning signal.",
    reward_stds,
)
essential_metric_names = [
    "reward", "reward_std", "frac_reward_zero_std", "kl", "entropy",
    "reward/task_success", "reward/no_answer_rate", "reward/invalid_trace_rate",
    "reward/correct_but_invalid_rate", "reward/shaping_adjustment_mean",
    "reward/nonnegative_invalid_rate", "calls/undercall_rate", "calls/exact_rate",
    "calls/overcall_rate", "groups/all_success_rate", "groups/all_failure_rate",
    "groups/mixed_rate", "groups/zero_reward_std_rate",
    "groups/easy_all_failure_rate", "groups/medium_all_failure_rate",
    "groups/hard_all_failure_rate", "groups/easy_zero_reward_std_rate",
    "groups/medium_zero_reward_std_rate", "groups/hard_zero_reward_std_rate",
    "trace/legal_progress_mean", "completion/truncated_rate",
    "tier/easy_task_success", "tier/medium_task_success", "tier/hard_task_success",
]
metric_summary = {}
for name in essential_metric_names:
    values = [entry[name] for entry in trainer.state.log_history if name in entry]
    if values:
        metric_summary[name] = {"min": min(values), "max": max(values), "last": values[-1]}

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(
    ADAPTER_DIR,
    selected_adapters=["default"],
    safe_serialization=True,
)
tokenizer.save_pretrained(ADAPTER_DIR)
shutil.copy2(SFT_IDS_PATH, ADAPTER_DIR / SFT_IDS_PATH.name)
shutil.copy2(GRPO_IDS_PATH, ADAPTER_DIR / GRPO_IDS_PATH.name)

package_names = [*PACKAGE_NAMES, "torch"]
run_config = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "started_at_utc": started_at,
    "ended_at_utc": ended_at,
    "seed": SEED,
    "base_model": BASE_ID,
    "base_revision": BASE_REVISION,
    "sft_adapter": SFT_ADAPTER_ID,
    "sft_adapter_revision": SFT_ADAPTER_REVISION,
    "sft_merged_before_grpo": False,
    "grpo_updates_existing_sft_adapter": True,
    "kl_reference": "frozen copy of initial SFT adapter",
    "generation_backend": BACKEND_NAME,
    "vllm_enabled": USE_VLLM,
    "num_rows": len(train_dataset),
    "input_sha256": {
        **observed_hashes,
        "source_notebook": sha256_file(SOURCE_NOTEBOOK),
        "semantic_module": sha256_file(Path(calculator_semantics.__file__)),
    },
    "gpu_name": gpu_name,
    "cuda_version": torch.version.cuda,
    "dtype": dtype_name,
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in package_names},
    "grpo_config": grpo_config.to_dict(),
    "reward_table": {
        "valid_correct": 2.0,
        "invalid_correct": 0.0,
        "valid_incorrect": 0.0,
        "invalid_incorrect": -1.0,
        "no_answer": -0.1,
    },
    "reward_shaping": {
        "mode": (
            "execution_binary"
            if EXECUTION_BINARY_REWARD
            else "strict_invalid" if STRICT_INVALID_REWARD
            else "per_call" if LEGAL_PROGRESS_BONUS_PER_CALL
            else "normalized" if LEGAL_PROGRESS_BONUS_MAX
            else "terminal_only"
        ),
        "legal_progress_bonus_max": LEGAL_PROGRESS_BONUS_MAX,
        "legal_progress_bonus_per_call": LEGAL_PROGRESS_BONUS_PER_CALL,
        "strict_invalid_reward": STRICT_INVALID_REWARD,
        "strict_valid_correct_reward": STRICT_VALID_CORRECT_REWARD,
        "execution_binary_reward": EXECUTION_BINARY_REWARD,
        "formula": (
            "2.0 if valid_trajectory and answer_correct else 0.0"
            if EXECUTION_BINARY_REWARD
            else "strict_invalid_matrix_v1" if STRICT_INVALID_REWARD
            else "per_call_bonus * min(legal_call_count, expected_call_count)"
            if LEGAL_PROGRESS_BONUS_PER_CALL
            else "max_bonus * min(legal_call_count, expected_call_count) / expected_call_count"
        ),
        "strict_invalid_upper_bound": -0.5 if STRICT_INVALID_REWARD else None,
        "no_answer_reward": 0.0 if EXECUTION_BINARY_REWARD else -1.2 if STRICT_INVALID_REWARD else -0.1,
    },
    "adapter_hashes": {
        "initial_default": initial_default_hash,
        "initial_ref": initial_ref_hash,
        "final_default": final_default_hash,
        "final_ref": final_ref_hash,
    },
    "train_metrics": train_result.metrics,
    "gpu_telemetry_summary": gpu_telemetry_summary,
    "hub_repo_id": HUB_REPO_ID if RUN_MODE == "full" else None,
    "hub_revision": None,
}
write_json_atomic(ADAPTER_DIR / "grpo_run_config.json", run_config)
write_json_atomic(RUN_DIR / "metrics.json", {
    "run_id": RUN_ID,
    "generation_backend": BACKEND_NAME,
    "train_metrics": train_result.metrics,
    "gpu_telemetry_summary": gpu_telemetry_summary,
    "reward_std": reward_stds,
    "essential_metric_summary": metric_summary,
    "last_log": trainer.state.log_history[-1],
    "status": "training_complete",
})
print({
    "train_metrics": train_result.metrics,
    "generation_backend": BACKEND_NAME,
    "gpu_telemetry_summary": gpu_telemetry_summary,
    "reward_std_min": min(reward_stds),
    "reward_std_max": max(reward_stds),
    "adapter_changed": final_default_hash != initial_default_hash,
    "reference_unchanged": final_ref_hash == initial_ref_hash,
    "saved_to": str(ADAPTER_DIR),
})

In [ ]:
# @title Prepare isolated Hub upload helper for a successful full run
UPLOAD_SCRIPT = Path("/content/upload_grpo_adapter.py")
if RUN_MODE == "full":
    UPLOAD_SCRIPT.write_text(f'''import json
import os
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import HfApi

repo_id = {HUB_REPO_ID!r}
adapter_dir = Path({str(ADAPTER_DIR)!r})
run_dir = Path({str(RUN_DIR)!r})
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
commit = api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(adapter_dir),
    commit_message="Qwen3 calculator GRPO-600 from pinned SFT adapter",
)
manifest = {{
    "repo_id": repo_id,
    "revision": commit.oid,
    "url": f"https://huggingface.co/{{repo_id}}/tree/{{commit.oid}}",
}}
(run_dir / "hub_revision.json").write_text(json.dumps(manifest, indent=2) + "\\n")
print("UPLOAD_COMPLETE", manifest)
''', encoding="utf-8")
    print("Full run passed. Restart the kernel, reinject .env, then execute:")
    print(
        "echo \"exec(open('/content/upload_grpo_adapter.py').read())\" "
        "| colab exec -s <session> --timeout 900"
    )
else:
    print("Smoke adapter remains local; Hub upload is intentionally disabled.")

In [ ]:
# @title Bundle lightweight artifacts for immediate download
import zipfile

bundle_path = Path(f"/content/{RUN_ID}-artifacts.zip")
artifact_paths = [
    ADAPTER_DIR / "grpo_run_config.json",
    RUN_DIR / "metrics.json",
    RUN_DIR / "trainer_state.json",
    SFT_IDS_PATH,
    GRPO_IDS_PATH,
]
revision_path = RUN_DIR / "hub_revision.json"
if revision_path.exists():
    artifact_paths.append(revision_path)
if UPLOAD_SCRIPT.exists():
    artifact_paths.append(UPLOAD_SCRIPT)

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_paths:
        assert path.is_file(), f"Missing required artifact: {path}"
        archive.write(path, arcname=path.name)
print(f"Created {bundle_path} ({bundle_path.stat().st_size:,} bytes)")
print("Download this bundle and the executed output notebook before stopping Colab:")
print(f'colab download -s <session> "{bundle_path}" "outputs/notebooks/{bundle_path.name}"')

## Colab execution gates

Run smoke first in a fresh T4 session:

```bash
colab sessions
colab version
colab new -s calc-grpo-smoke --gpu T4
colab status -s calc-grpo-smoke
echo "from pathlib import Path; Path('/content/data').mkdir(parents=True, exist_ok=True)" | colab exec -s calc-grpo-smoke
colab upload -s calc-grpo-smoke notebooks/grpo_qwen3_calculator.ipynb /content/grpo_qwen3_calculator.ipynb
colab upload -s calc-grpo-smoke scripts/calculator_semantics.py /content/calculator_semantics.py
colab upload -s calc-grpo-smoke data/calculator_qwen3 /content/data/calculator_qwen3
colab upload -s calc-grpo-smoke data/calculator_qwen3_sft200_ids.txt /content/data/calculator_qwen3_sft200_ids.txt
colab upload -s calc-grpo-smoke data/calculator_qwen3_grpo600_ids.txt /content/data/calculator_qwen3_grpo600_ids.txt
colab upload -s calc-grpo-smoke .env /content/.env
colab exec -s calc-grpo-smoke -f .cursor/skills/colab-cli/scripts/load-env.py --timeout 60
colab exec -s calc-grpo-smoke -f notebooks/grpo_qwen3_calculator.ipynb --timeout 7200
```

The smoke run is acceptable only if reward tests pass, reward standard deviation is nonzero, the trainable adapter hash changes, the frozen reference hash does not change, and all metrics remain finite. Download the bundle and `notebooks/grpo_qwen3_calculator_output.ipynb`, inspect them locally, then stop the session.

Keep `USE_VLLM=0` on T4. To benchmark colocated vLLM, allocate L4/A100, set `USE_VLLM=1`, and require a separate smoke pass before any full run. Do not assume vLLM is faster until comparing `train_runtime`, `train_steps_per_second`, and peak memory against the Transformers smoke.

For the full run, use a fresh session and inject `RUN_MODE=full` through `.env`. After training, preserve the executed notebook and artifacts. Restart the kernel before running the generated upload helper, reinject `.env`, run the helper from `/content`, download `hub_revision.json`, and verify the immutable Hub revision. Finally stop the session and confirm `colab sessions` shows no unintended runtime.

Do not load the eval split during training. Select checkpoints later with the existing greedy eval harness; use the test split once for the final pinned adapter.